nov 19th

In [89]:
# import xarray as xr
from scipy.interpolate import NearestNDInterpolator
from scipy.interpolate import griddata
import pandas as pd
import numpy as np
# from scipy.spatial import cKDTree
import matplotlib.pyplot as plt

### bringing in age

In [ ]:
dt = pd.read_csv("..\\data\\nov19\\long_lat_age.csv", skiprows=1,
                 names=['long', 'lat', 'age'])
print(dt)

       long   lat        age
0    -133.0  52.9  10.251687
1    -132.9  52.9   9.945641
2    -133.0  52.8  10.205028
3    -132.9  52.8   9.673500
4    -132.8  52.8   9.405209
...     ...   ...        ...
8465 -124.9  40.0  15.446379
8466 -124.8  40.0  14.237822
8467 -124.7  40.0  15.990977
8468 -124.6  40.0  14.295403
8469 -124.5  40.0  15.872827

[8470 rows x 3 columns]


In [91]:
print(dt['long'].min(), dt['long'].max(), dt['lat'].min(), dt['lat'].max())

-133.0 -124.5 40.0 52.9


### bringing in z (picks) in ms

In [92]:
dz = pd.read_csv("..\\data\\nov19\\long_lat_z.csv", skiprows=1, names=['long', 'lat', 'z'])
print(dz)

            long        lat         z
0    -127.072914  47.572946  4586.444
1    -127.072996  47.572970  4586.128
2    -127.073076  47.572985  4585.567
3    -127.073144  47.573000  4585.362
4    -127.073225  47.573015  4586.053
...          ...        ...       ...
7456 -127.706619  47.701780  4045.076
7457 -127.706701  47.701795  4043.890
7458 -127.706782  47.701809  4042.803
7459 -127.706863  47.701824  4040.274
7460 -127.706931  47.701839  4036.368

[7461 rows x 3 columns]


In [93]:
print(dz['long'].min(), dz['long'].max(), dz['lat'].min(), dz['lat'].max())

-127.7069313 -127.0729138 47.57294553 47.70183923


In [94]:
trim_dt = dt[dt['long'].between(-127.8, -127) & dt['lat'].between(47.4, 47.8)]

In [95]:
print(len(trim_dt))

45


interpolate age [dt] at each z [dz] point
- we have more z [dz] than age [dt] after filtering

> griddata(points, values, data to interp --> [scipy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.griddata.html)

In [96]:
age_interp = griddata((trim_dt[['long','lat']].values), trim_dt['age'].values, dz[['long','lat']].values, method='linear')
age_nan_fill = griddata((trim_dt[['long','lat']].values), trim_dt['age'].values, dz[['long','lat']].values, method='nearest')

In [97]:
dz['age'] = np.where(np.isnan(age_interp), age_nan_fill, age_interp)

print(dz)

            long        lat         z       age
0    -127.072914  47.572946  4586.444  4.983188
1    -127.072996  47.572970  4586.128  4.982962
2    -127.073076  47.572985  4585.567  4.982734
3    -127.073144  47.573000  4585.362  4.982544
4    -127.073225  47.573015  4586.053  4.982316
...          ...        ...       ...       ...
7456 -127.706619  47.701780  4045.076  3.476296
7457 -127.706701  47.701795  4043.890  3.476080
7458 -127.706782  47.701809  4042.803  3.475864
7459 -127.706863  47.701824  4040.274  3.475648
7460 -127.706931  47.701839  4036.368  3.475463

[7461 rows x 4 columns]


double check no nan

In [101]:
np.where(np.isnan(dz))

(array([], dtype=int64), array([], dtype=int64))

In [105]:
dz[np.isclose(dz['age'], 4, atol=0.1)]  

,long,lat,z,age
4363,-127.449412,47.649997,4162.572,4.100023
4364,-127.449493,47.650012,4163.100,4.099902
4365,-127.449574,47.650027,4163.224,4.099781
4366,-127.449655,47.650042,4162.798,4.099659
4367,-127.449737,47.650057,4161.981,4.099538
...,...,...,...,...
5110,-127.512639,47.662787,3955.875,3.900994
5111,-127.512720,47.662802,3953.698,3.900748
5112,-127.512788,47.662818,3951.821,3.900533
5113,-127.512869,47.662832,3949.833,3.900287


### downloading asf

for the entire dataset

In [106]:
dz.to_csv("..\\created_data\\nov19_long_lat_age_z.csv", index=False)

for filt data

In [ ]:
# filt_dg = dt[dt['long'].between(-129, -127) & dt['lat'].between(45, 47)]

# filt_dg.to_csv("..\\created_data\\filt_lola_grav_age_q_z_bath.csv", index=False)

            long        lat        FA
4891 -128.916670  45.100001 -8.326332
4892 -128.916670  45.216668 -4.583536
4893 -128.916670  45.333334 -5.879004
4894 -128.916670  45.450001 -4.933280
4895 -128.916670  45.566668 -5.931483
...          ...        ...       ...
7111 -127.050003  46.500001 -5.146489
7112 -127.050003  46.616668 -9.314369
7113 -127.050003  46.733334 -7.298386
7114 -127.050003  46.850001 -6.077028
7115 -127.050003  46.966668 -7.009515

[289 rows x 3 columns]
